# IEEE-CIS Fraud Detection Models Data

In [7]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ieee-fraud-detection')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/ieee-fraud-detection


In [8]:
# Core Python
import os
import gc
import warnings
warnings.filterwarnings("ignore")

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

# Metrics
from sklearn.metrics import roc_auc_score, classification_report

# Models
import lightgbm as lgb
import xgboost as xgb


In [9]:
#load all the csv files
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")

test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


In [10]:
train_transaction_file_path = ("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
#train_transaction = pd.read_csv("train_transaction_file_path")
train_transaction.head()

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
#checking the dataset shape
train_transaction.shape, train_identity.shape
test_transaction.shape, test_identity.shape


((506691, 393), (141907, 41))

In [12]:
#first row preview
train_transaction.head()
train_identity.head()

,TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663586,-45.0,280290.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
1,3663588,0.0,3579.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,24.0,1280x720,match_status:2,T,F,T,T,mobile,LGLS676 Build/MXB48T
2,3663597,-5.0,185210.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,...,ie 11.0 for tablet,NaN,NaN,NaN,F,T,T,F,desktop,Trident/7.0
3,3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
4,3663602,-95.0,328680.0,NaN,NaN,7.0,-33.0,NaN,NaN,NaN,...,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-G9650 Build/R16NW


In [13]:
#previewing the missing to 20 values
train_transaction.isnull().sum().head(20)

TransactionID          0
TransactionDT          0
TransactionAmt         0
ProductCD              0
card1                  0
card2               8654
card3               3002
card4               3086
card5               4547
card6               3007
addr1              65609
addr2              65609
dist1             291217
dist2             470255
P_emaildomain      69192
R_emaildomain     370821
C1                     3
C2                     3
C3                     3
C4                     3
dtype: int64

In [14]:
#marge the transaction and the identity table
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")


In [15]:
#verify merge dataset
train.shape, test.shape

((506691, 433), (506691, 433))

In [16]:
# we preview the head
train.head()

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# we check the existing target
"isFraud" in train.columns

False

In [18]:
# check missing values
train.isnull().sum().sort_values(ascending=False).head(20)


id-24    501951
id-25    501652
id-26    501644
id-21    501632
id-08    501632
id-07    501632
id-27    501629
id-23    501629
id-22    501629
dist2    470255
id-18    455816
D7       446558
id-04    440210
id-03    440210
D12      437437
id-30    436032
id-33    436020
id-32    436020
id-14    435334
id-34    434516
dtype: int64

In [19]:
# separate features and target
y = train["isFraud"]
train = train.drop(columns=["isFraud"])


KeyError: 'isFraud'